In [1]:
import pandas as pd
import numpy as np

In [4]:
claims_data = pd.read_excel("../../data/raw/ClaimDetails_for_distribution.xlsx")
adjuster_data = pd.read_excel("../../data/raw/CatRosterReportDayOf_for_distribution_cleaned.xlsx")

# Quick check
adjuster_data.head()

,TIES Id,Org Group,Department,Location,HR Job Title,ERT Champion Ind,ERT Champion,WFM Tour Supervisor Name,Preferred Role,Availability,...,"Subro Claim Rep, Recovery",Subro SIU Investigator,Subro Unit Mgr,Videographer,WC Clm Rslv Case Mgr,WC Critical Clm Spcl,WC investigative Case Mgr,WC Medical Case Mgr,WC RTW Case Mgr,WC Unit Mgr
0,47334245,Upper Midwest,Department_35,St. Paul,"Claim Rep Trainee, Outside Property",,NaN,NaN,Property Outside Claim Rep,NaN,...,0,0,0,0,0,0,0,0,0,0
1,43992727,Upper Midwest,Department_35,West Des Moines-Jordan Creek,"Claim Rep Trainee, Outside Property",N,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
2,45899542,Upper Midwest,Department_35,St. Paul,"Claim Rep Trainee, Outside Property",N,NaN,NaN,Property Outside Claim Rep,NaN,...,0,0,0,0,0,0,0,0,0,0
3,41040719,Upper Midwest,Department_35,Overland Pk-Kansas City-132 St,"Claim Rep Trainee, Outside Property",N,NaN,NaN,Property Outside Claim Rep,NaN,...,0,0,0,0,0,0,0,0,0,0
4,92150926,Upper Midwest,Department_35,Maryland Heights-St. Louis,"Claim Rep Trainee, Outside Property",N,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# ============================================================
# ADJUSTER ELIGIBILITY FILTERING LOGIC
# ============================================================
# This script:
# 1️⃣ Generates a minimum Required Skill Level from CAT Severity
# 2️⃣ Filters adjusters to Available + Non-Trainees
# 3️⃣ Counts how many adjusters meet minimum skill requirements
# 4️⃣ Uses Division-specific skill logic:
#       - BI → CL Skill Level
#       - PI → PL Skill Level
# 5️⃣ Required Skill Level acts as a MINIMUM (>= logic)
# ============================================================

In [5]:
# ============================================================
# 1️⃣ Generate Required Skill Level (Minimum Required)
# ============================================================
# Maps CAT Severity Code to the minimum skill level required.
# Higher severity = higher minimum skill requirement.
# If severity is missing, Required Skill remains NA.

# Create Required Skill Level column
claims_data["Required Skill Level"] = np.select(
    [
        claims_data["CAT Severity Code"] == 5,
        claims_data["CAT Severity Code"] == 4,
        claims_data["CAT Severity Code"] == 3,
        claims_data["CAT Severity Code"].isin([1, 2])
    ],
    [4, 3, 2, 1],
    default=np.nan  # Leave missing if severity missing
)

In [7]:
# ============================================================
# 2️⃣ Filter Adjuster Pool
# ============================================================
# Keeps only:
# ✔ Adjusters currently Available
# ✔ Adjusters who are not trainees
#     (Trainees assumed to have skill level 0 in both CL and PL)

adjuster_pool = adjuster_data[
    (adjuster_data["Current Status"] == "Available") &
    (
        (adjuster_data["CL Skill Level"].fillna(0) > 0) |
        (adjuster_data["PL Skill Level"].fillna(0) > 0)
    )
].copy()

In [8]:
# ============================================================
# 3️⃣ Count Eligible Adjusters Per Claim
# ============================================================
# For each claim:
# - Look at its Division (BI or PI)
# - Use the appropriate skill column
# - Count adjusters whose skill >= Required Skill Level
#
# IMPORTANT:
# Required Skill Level is a MINIMUM threshold.
# Example:
#   If Required Skill = 2
#   Then adjusters with skill 2, 3, 4, 5 all qualify.

def count_eligible(row):
    
    required_skill = row["Required Skill Level"]
    
    # If Required Skill missing → no eligible adjusters
    if pd.isna(required_skill):
        return 0
    
    # BI Claims → use CL Skill Level
    if row["Division"] == "BI":
        return (
            adjuster_pool["CL Skill Level"] >= required_skill
        ).sum()
    
    # PI Claims → use PL Skill Level
    elif row["Division"] == "PI":
        return (
            adjuster_pool["PL Skill Level"] >= required_skill
        ).sum()
    
    # If Division missing or unrecognized
    else:
        return 0


# Apply row-wise (equivalent to rowwise() in dplyr)
claims_data["Num_Eligible"] = claims_data.apply(
    count_eligible,
    axis=1
)

In [ ]:
# ============================================================
# OUTPUT
# ============================================================
# claims_data now contains:
#   Required Skill Level  → minimum skill needed per claim
#   Num_Eligible          → number of available, non-trainee
#                           adjusters meeting minimum skill
#
# No constraints included for:
#   - Travel
#   - Resource Type
#
# Eligibility is now purely:
#   Available + Non-Trainee + Skill >= Minimum Required
# ============================================================

In [9]:
claims_data[["CAT Severity Code",
             "Required Skill Level",
             "Division",
             "Num_Eligible"]].head()

,CAT Severity Code,Required Skill Level,Division,Num_Eligible
0,3,2.0,PI,534
1,5,4.0,PI,104
2,5,4.0,PI,104
3,2,1.0,PI,616
4,3,2.0,PI,534
